# Free-Lunch — a quantitative teardown 🔬
### Beta-neutral construction · the 2.78× leverage · the financing sweep · gross-vs-market · decay

![Signal: Weak](https://img.shields.io/badge/Signal-Weak-dab617?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Free lunch?: Busted](https://img.shields.io/badge/Free_lunch%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). We build BAB on a tradable ETF cross-section and price the leverage it depends on.

> ⚠️ **Not investment advice.** 13 ETFs + SPY, daily, 2000–2026 (Yahoo). ETFs keep it survivorship-free at the cost of coarser beta dispersion. Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (free_lunch/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from free_lunch import data, strategy as st
assets, market = data.fetch_etf_panel()       # cache-first; built by examples/verify.py --fetch
gross, lev = st.bab_returns(assets, market, return_leverage=True)
mkt = st.market_monthly(market).loc[gross.index]


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **Weak** | gross Sharpe 0.47 < market 0.59 |
| Tradability | **Mirage** | 2.78× leverage; net Sharpe 0.47→0.20→0.02 |
| Free lunch? | **Busted** | the premium is the leverage rent; decayed 0.70→0.28 |

> 💡 *In plain words:* the headline lives entirely in a free-borrowing assumption.

## 1 · The claim, steelmanned

- **H₁:** the beta-neutral low-minus-high book earns a positive gross premium.
- **H₂ (the pitch):** it beats the market risk-adjusted after the financing its leverage requires.
- **H₃:** the edge is stable.

## 2 · So what? — what rides on each

If H₂ holds it's a low-risk market-beater. If only H₁ holds, the premium is real but captured by the lender, not you.

## 3 · How we'd know — the protocol

Trailing betas → median split → lever each leg to beta 1 (record the long-leg leverage) → gross BAB → **financing sweep** → vs market → decade split.

## 4 · The teardown

### 4.1 The construction and its leverage

In [2]:
print(f'avg long-leg leverage = {lev:.2f}x  (low-beta leg β≈{1/lev:.2f} levered to 1)')
display(pd.DataFrame({'BAB gross':st.summary(gross),'SPY':st.summary(mkt)}).T[['cagr','sharpe','vol_ann','max_drawdown']].round(3))

avg long-leg leverage = 2.78x  (low-beta leg β≈0.36 levered to 1)


,cagr,sharpe,vol_ann,max_drawdown
BAB gross,0.081,0.472,0.216,-0.453
SPY,0.081,0.590,0.152,-0.508


> 💡 *In plain words:* even before any cost, BAB's Sharpe trails the market. **H₁ holds only weakly** — there's a tilt, but it doesn't clear the bar.

### 4.2 The financing sweep — where the edge goes

In [3]:
rates=np.linspace(0,0.06,13)
tbl=pd.DataFrame({'financing %':rates*100,
  'BAB net Sharpe':[st.summary(st.bab_returns(assets,market,financing_ann=r,borrow_ann=r/6))['sharpe'] for r in rates]})
display(tbl.round(3).set_index('financing %'))
print('market Sharpe for reference:', round(st.summary(mkt)['sharpe'],2))

,BAB net Sharpe
financing %,
0.0,0.472
0.5,0.428
1.0,0.383
1.5,0.338
2.0,0.294
2.5,0.249
3.0,0.204
3.5,0.160
4.0,0.115


market Sharpe for reference: 0.59


> 💡 *In plain words:* at a plausible 3% the Sharpe is 0.20; at 5% it's ~0. **H₂ rejected** — the leverage rent is the premium.

### 4.3 Decay

In [4]:
for lab,sl in [('1999-2012',gross.loc[:'2012']),('2013-on',gross.loc['2013':])]:
    print(f'{lab}: gross Sharpe {st.summary(sl)["sharpe"]:.2f}')

1999-2012: gross Sharpe 0.70
2013-on: gross Sharpe 0.28


> 💡 *In plain words:* 0.70 → 0.28, the post-publication fade on top of the financing problem. **H₃ rejected.**

## 5 · The verdict

H₁ weak, H₂ and H₃ rejected → Signal `WEAK`, Tradability `MIRAGE`, free lunch `BUSTED`.

## 6 · Could you trade it?

Only with cheap, permanent ~1.8× leverage — the very thing the anomaly says constrained investors *lack*. Price your own borrow honestly and there is no edge.

## 7 · Going further

Forks: (a) a single-stock universe for finer beta dispersion (more headroom gross, same financing kill); (b) the international BAB variant; (c) overlay a borrow rate that *moves* with Fed funds. Backlog: [`docs/pwb_strategies_inventory.md`](../../../docs/pwb_strategies_inventory.md). Mechanism twin: [Study 30 House-Edge](../../30-house-edge/).